In [1]:
import pandas as pd
import mygene
import os

In [6]:
import sys
sys.path.append("..")

from choose_protein_coding import list_of_protein_coding_genes

In [7]:
gene_info_file = "../../data/gene_info.csv"

In [8]:
filepath = '../../data/GTEx'
filename = 'gene_tpm_v11_bladder'
df_gtex = pd.read_csv(f"{filepath}/{filename}.gct", sep='\t', skiprows=2)
# delete ensemblid versioning
df_gtex['Name'] = df_gtex['Name'].str.split('.').str[0]

df_gtex.head()

,Name,Description,GTEX-N7MS-2126-SM-GRR2X,GTEX-N7MT-1826-SM-GQ1C9,GTEX-NPJ8-1126-SM-GNTAI,GTEX-O5YT-1926-SM-H6Q82,GTEX-OHPK-1926-SM-H7OF2,GTEX-OHPL-1926-SM-HAV18,GTEX-OHPM-1926-SM-H7OG1,GTEX-OIZF-1926-SM-7PBZS,...,GTEX-T5JW-1026-SM-EZ6LR,GTEX-T6MN-2226-SM-EVYAM,GTEX-T6MO-0926-SM-GPRWN,GTEX-T8EM-1726-SM-GPRWO,GTEX-TKQ2-0526-SM-GRR15,GTEX-TMMY-1526-SM-4DXST,GTEX-U3ZH-0826-SM-H6Q7I,GTEX-U3ZM-0826-SM-4DXU6,GTEX-U3ZN-1226-SM-4DXUD,GTEX-U4B1-1226-SM-4DXT7
0,ENSG00000290825,DDX11L16,0.00000,0.000000,0.008595,0.000000,0.017163,0.016102,0.015161,0.016748,...,0.015152,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.028452
1,ENSG00000223972,DDX11L1,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000000
2,ENSG00000310526,WASH7P,1.84073,8.343940,6.056380,5.239430,0.580347,2.797340,2.326170,3.015620,...,3.893930,2.861550,3.271850,2.97508,3.28538,1.93326,6.70663,2.25086,4.12759,2.369130
3,ENSG00000243485,MIR1302-2HG,0.00000,0.038021,0.014900,0.026994,0.000000,0.027913,0.026281,0.000000,...,0.026265,0.000000,0.086383,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.049320
4,ENSG00000237613,FAM138A,0.00000,0.000000,0.000000,0.033700,0.000000,0.000000,0.000000,0.036243,...,0.000000,0.022609,0.000000,0.02322,0.00000,0.00000,0.00000,0.00000,0.00000,0.000000


In [3]:
# keep only TCGA genes - load TCGA example file
filename = '../../data/raw_tsv_data/TCGA-GBM.star_tpm.tsv'
df_tcga = pd.read_csv(filename, sep='\t')

# delete ensemblid versioning
df_tcga['Ensembl_ID'] = df_tcga['Ensembl_ID'].str.split('.').str[0]
df_tcga.head()

,Ensembl_ID,TCGA-27-2521-01A,TCGA-19-1390-01A,TCGA-27-1830-01A,TCGA-32-1970-01A,TCGA-06-0190-01A,TCGA-06-0190-02A,TCGA-02-2486-01A,TCGA-14-0790-01B,TCGA-14-0789-01A,...,TCGA-27-2519-01A,TCGA-26-5134-01A,TCGA-06-0675-11A,TCGA-06-1804-01A,TCGA-14-1829-01A,TCGA-06-0184-01A,TCGA-32-2638-01A,TCGA-12-3652-01A,TCGA-28-5209-01A,TCGA-76-4925-01A
0,ENSG00000000003,2.783163,4.502031,6.164816,5.940879,6.149881,5.386263,6.818908,6.559516,5.845139,...,6.228588,5.399151,3.284174,5.722884,7.908395,6.523104,6.747138,6.949359,6.377500,6.316540
1,ENSG00000000005,1.203076,0.066537,0.673918,0.899794,0.366588,1.093695,0.659377,0.840926,0.092884,...,1.225892,0.585347,0.388796,0.412727,3.007052,0.505078,1.388906,0.425674,0.626392,0.496718
2,ENSG00000000419,6.869697,5.901299,6.227456,5.746119,6.351695,6.433978,6.693888,6.274832,5.687909,...,6.574426,5.576350,5.762248,6.065012,7.120749,6.106935,6.559345,6.912564,6.399855,7.415961
3,ENSG00000000457,2.990828,2.863225,2.859135,2.728552,2.551934,2.350441,2.794790,2.699751,2.529646,...,2.489389,2.603976,2.501261,2.616640,2.589883,2.350271,2.846473,2.152768,2.952129,2.950095
4,ENSG00000000460,2.421937,2.992080,2.234685,2.321986,2.991481,2.697752,2.018385,2.290572,1.792730,...,1.942796,2.511392,0.914488,2.080009,2.832566,1.948788,2.313217,2.616734,2.900258,2.785320


In [4]:
# make protein coding lists from knowledge file

df = pd.read_csv('../../data/gene_info_table.csv', index_col=0)

duplicate_check = df.groupby('ensembl_id')['gene_type'].nunique()
ids_with_multiple_types = duplicate_check[duplicate_check > 1].index.tolist()

if ids_with_multiple_types:
    print(f"Znaleziono duplikaty z różnymi typami dla ID: {ids_with_multiple_types}")
    for gene_id in ids_with_multiple_types:
        names = df[df['ensembl_id'] == gene_id]['gene_name'].unique()
        print(f"  - Gen {gene_id} ({names}) występuje z wieloma typami. Zostanie zachowany jako protein_coding.")

protein_coding_df = df[df['gene_type'] == 'protein_coding'].copy()
protein_coding_df = protein_coding_df.drop_duplicates(subset=['ensembl_id'])

ensembl_list = protein_coding_df['ensembl_id'].tolist()
gene_name_list = protein_coding_df['gene_name'].tolist()

# Printing results
print(f"Number of protein coding ensembl: {len(ensembl_list)}")
print(f"Number of protein coding gene_name: {len(gene_name_list)}")
print("First 5 ID:", ensembl_list[:5])
print("First 5 nazw:", gene_name_list[:5])

Number of protein coding ensembl: 23394
Number of protein coding gene_name: 23394
First 5 ID: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460']
First 5 nazw: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']


In [5]:
valid_ids = set(df_tcga['Ensembl_ID'].unique())
df_gtex_filtered = df_gtex[df_gtex['Name'].isin(valid_ids)].copy()

# Printing results:
print(f"Gene number before filtering: {len(df_gtex)}")
print(f"Gene number after filtering: {len(df_gtex_filtered)}")

lost_genes = len(df_gtex) - len(df_gtex_filtered)
if lost_genes > 0:
    print(f"Attention: {lost_genes} gened from GTEx were lost.")

    gtex_ids = set(df_gtex['Name'].unique())
    missing_ids = gtex_ids - valid_ids
    missing_protein_coding = [gene_id for gene_id in missing_ids if gene_id in ensembl_list]
    print(f"Out of which {len(missing_protein_coding)} were protein coding genes")

Gene number before filtering: 74628
Gene number after filtering: 55751
Attention: 18877 gened from GTEx were lost.
Out of which 0 were protein coding genes


In [7]:
# filtering of gtex df
df_gtex = df_gtex[df_gtex['Name'].isin(valid_ids)].reset_index(drop=True)

In [18]:
# choose protein coding in the same way as in TCGA data

def list_of_protein_coding_genes(gene_list: list):
    genes = pd.Index(gene_list)

    # load gene info data, drop duplacates
    gene_info_original = pd.read_csv('../../data/gene_info_table.csv')
    gene_info = gene_info_original.drop(columns=['ensembl_id', 'Unnamed: 0'])
    gene_info = gene_info.drop_duplicates()

    # choose 1 gene_type for ambiguous genes
    gene_info["priority"] = (gene_info["gene_type"] == "protein_coding").astype(int)
    gene_info = gene_info.sort_values(
        ["gene_name", "priority"],
        ascending=[True, False]
    )
    gene_info = gene_info.drop_duplicates("gene_name")
    gene_info = gene_info.drop(columns="priority")

    gene_type_map = gene_info.set_index("gene_name")["gene_type"]
    mapped_types = genes.map(gene_type_map)
    missing_genes = genes[mapped_types.isna()]

    # another gene_info for missed genes
    gene_ensembl = pd.read_csv('../../data/gene_info.csv')
    symbol_to_ens_map = gene_ensembl.set_index("feature_name")["feature_id"]
    missing_ens = missing_genes.map(symbol_to_ens_map)
    gene_info_ens = gene_info_original.drop(columns=['gene_name', 'Unnamed: 0'])
    gene_info_ens = gene_info_ens.drop_duplicates()

    # choose 1 gene_type for ambiguous genes
    gene_info_ens["priority"] = (gene_info_ens["gene_type"] == "protein_coding").astype(int)
    gene_info_ens = gene_info_ens.sort_values(
        ["ensembl_id", "priority"],
        ascending=[True, False]
    )
    gene_info_ens = gene_info_ens.drop_duplicates("ensembl_id")
    gene_info_ens = gene_info_ens.drop(columns="priority")

    # creat mapping
    gene_type_map_ens = gene_info_ens.set_index("ensembl_id")["gene_type"]
    missing_types_from_ens = missing_ens.map(gene_type_map_ens)

    mapped_types = pd.Series(mapped_types, index=genes)
    mapped_types.loc[missing_genes] = missing_types_from_ens.values
    protein_coding_genes = mapped_types[mapped_types == "protein_coding"].index.tolist()

    return protein_coding_genes


In [12]:
df = df_gtex.drop(columns=['Description'])
df = df.T
df.columns = df.iloc[0]
df = df.iloc[1:]

In [15]:
features = pd.read_csv(gene_info_file)
id_to_symbol = dict(zip(features["feature_id"], features["feature_name"]))
df = df.rename(columns=id_to_symbol)

In [16]:
df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

In [19]:
gene_list = df.columns
protein_coding = list_of_protein_coding_genes(gene_list)

In [20]:
df_filtered = df[df.columns.intersection(protein_coding)]

In [21]:
df_filtered = df_filtered[~df_filtered.index.duplicated(keep="first")]

In [22]:
df_filtered

Name,OR4F5,OR4F29,OR4F16,SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,...,MT-CO2,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB
GTEX-N7MS-2126-SM-GRR2X,0.052071,0.0,0.036294,17.9641,61.3032,23.369,22.1573,0.509546,52.7262,80.2589,...,39381.5,22181.0,37565.1,42830.8,14096.2,21647.3,43243.6,3648.08,2333.95,20282.9
GTEX-N7MT-1826-SM-GQ1C9,0.015801,0.0,0.0,25.4047,71.7933,26.0032,31.7554,2.73289,136.91,92.6908,...,18813.9,13811.3,23997.2,17329.2,15075.9,8127.98,15801.5,2210.57,1609.55,11033.6
GTEX-NPJ8-1126-SM-GNTAI,0.006192,0.034528,0.017264,16.6913,103.021,33.7339,1.10142,0.507295,20.7001,42.6828,...,18232.8,7583.98,13293.7,13516.8,11320.3,6152.37,10094.7,2305.28,1535.87,9737.0
GTEX-O5YT-1926-SM-H6Q82,0.044874,0.0,0.0,18.1108,77.2473,21.4053,31.9275,0.980355,38.4066,39.5131,...,28233.3,18541.1,31691.2,20318.6,15141.9,20123.5,40007.7,10685.4,13128.1,19436.4
GTEX-OHPK-1926-SM-H7OF2,0.049456,0.0,0.068943,0.765342,67.3625,4.7378,0.571072,0.101293,5.44668,17.723,...,39193.0,29338.6,22912.4,38540.5,13120.4,23631.6,41193.5,15046.5,20429.7,23808.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTEX-TMMY-1526-SM-4DXST,0.160172,0.0,0.127592,2.32289,62.6509,15.6558,2.96823,0.312435,7.02723,21.1936,...,28547.2,15328.0,26704.1,24046.0,9986.17,12985.0,30673.2,5625.11,6586.04,20479.5
GTEX-U3ZH-0826-SM-H6Q7I,0.013543,0.0,0.0,31.1974,74.4459,27.504,29.3605,1.83692,58.1958,60.7576,...,30931.2,22386.7,32772.9,34708.7,18065.8,15083.9,34475.9,4845.58,5504.01,26497.5
GTEX-U3ZM-0826-SM-4DXU6,0.040382,0.112587,0.0,2.88296,62.5945,11.1158,2.38107,0.330833,23.9394,19.1466,...,34615.9,23003.2,40178.8,39289.5,15971.4,19283.8,42613.1,13378.7,16907.1,27396.6
GTEX-U3ZN-1226-SM-4DXUD,0.025035,0.0,0.0,26.6549,76.0476,23.4504,66.7476,2.5524,99.6995,73.6658,...,30314.3,17304.7,30975.6,19946.2,17819.7,15418.4,38003.2,12835.3,17888.7,24090.1


In [26]:
save_dir = f"../../data/GTEx/processed"

os.makedirs(save_dir, exist_ok=True)
# Zapisujemy gotowy plik, np. z dopiskiem '_processed'
save_path = os.path.join(save_dir, f"{filename}_processed.csv")
df_filtered.to_csv(save_path)
print(f"Zapisano przetworzone dane GTEx do: {save_path}")

Zapisano przetworzone dane GTEx do: ../../data/GTEx/processed/gene_tpm_v11_bladder_processed.csv


In [ ]:

mg = mygene.MyGeneInfo()

In [9]:
ensembl_ids = ["ENSG00000290825", "ENSG00000223972", "ENSG00000227159"] 

results = mg.querymany(ensembl_ids, 
                       scopes='ensembl.gene', 
                       fields='symbol,type_of_gene', # W mygene biotyp jest pod 'type_of_gene'
                       species='human')

for res in results:
    print(f"ID: {res['query']} | Symbol: {res.get('symbol')} | Typ: {res.get('type_of_gene')}")

ID: ENSG00000290825 | Symbol: DDX11L1 | Typ: pseudo
ID: ENSG00000223972 | Symbol: DDX11L1 | Typ: None
ID: ENSG00000227159 | Symbol: DDX11L16 | Typ: None


Liczba genów protein coding ensembl: 23394
Liczba genów protein coding gene_name: 23394
Pierwsze 5 ID: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460']
Pierwsze 5 nazw: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']


In [14]:
# 1. Tworzymy kolumny sprawdzające obecność na listach
# Zakładamy, że df_gtex ma już 'Name' po usunięciu kropek
df_gtex['is_protein_ensembl'] = df_gtex['Name'].isin(ensembl_list)
df_gtex['is_protein_name'] = df_gtex['Description'].isin(gene_name_list)

# 2. Sprawdzamy, gdzie wyniki się NIE pokrywają (mismatch)
mismatch = df_gtex[df_gtex['is_protein_ensembl'] != df_gtex['is_protein_name']]

# 3. Wyświetlamy statystyki
print(f"Liczba genów zgodnych (oba True): {len(df_gtex[(df_gtex['is_protein_ensembl']) & (df_gtex['is_protein_name'])])}")
print(f"Liczba genów zgodnych (oba False): {len(df_gtex[(~df_gtex['is_protein_ensembl']) & (~df_gtex['is_protein_name'])])}")
print(f"Liczba niezgodności (mismatch): {len(mismatch)}")

if len(mismatch) > 0:
    print("\nPrzykłady niezgodności (pierwsze 10):")
    print(mismatch[['Name', 'Description', 'is_protein_ensembl', 'is_protein_name']].head(10))

Liczba genów zgodnych (oba True): 17798
Liczba genów zgodnych (oba False): 55030
Liczba niezgodności (mismatch): 1800

Przykłady niezgodności (pierwsze 10):
                Name Description  is_protein_ensembl  is_protein_name
64   ENSG00000187642       PERM1                True            False
96   ENSG00000184163    C1QTNF12                True            False
104  ENSG00000127054      INTS11                True            False
116  ENSG00000224870  MRPL20-AS1                True            False
132  ENSG00000215014   SSU72-AS1                True            False
134  ENSG00000228594      FNDC10                True            False
195  ENSG00000157870      PRXL2B                True            False
418  ENSG00000179840  PIK3CD-AS1                True            False
445  ENSG00000175279       CENPS                True            False
481  ENSG00000204624       DISP3                True            False


In [15]:
# 1. Przygotowanie unikalnych ID z GTEx (bez kropek)
ensembl_ids = df_gtex['Name'].unique().tolist()

# 2. Zapytanie do MyGene (pobieramy symbol i typ genu)
mg = mygene.MyGeneInfo()
print(f"Pobieranie danych dla {len(ensembl_ids)} genów z MyGene.info...")

# querymany jest bardzo szybkie dla dużych list
records = mg.querymany(ensembl_ids, 
                       scopes='ensembl.gene', 
                       fields='symbol,type_of_gene', 
                       species='human', 
                       as_dataframe=True)

# 3. Przygotowanie wyników z MyGene
# Mapujemy typy na Twoją kategorię 'protein_coding'
# MyGene używa 'protein-coding' (z myślnikiem), więc musimy to ujednolicić
records['is_protein_mygene'] = records['type_of_gene'] == 'protein-coding'

# 4. Połączenie z df_gtex
# MyGene zwraca DataFrame, gdzie index to oryginalne zapytanie (ID)
df_gtex = df_gtex.merge(records[['is_protein_mygene']], 
                        left_on='Name', 
                        right_index=True, 
                        how='left')

# 5. Dodanie Twoich poprzednich kolumn (na podstawie lokalnych list)
df_gtex['is_protein_ensembl_local'] = df_gtex['Name'].isin(ensembl_list)

# 6. Porównanie: Lokalna lista vs MyGene Online
mismatch_local_vs_online = df_gtex[df_gtex['is_protein_mygene'] != df_gtex['is_protein_ensembl_local']]

print(f"\n--- Statystyki Porównawcze ---")
print(f"Zgodność z MyGene (Protein): {df_gtex['is_protein_mygene'].sum()}")
print(f"Zgodność z Twoją listą: {df_gtex['is_protein_ensembl_local'].sum()}")
print(f"Liczba rozbieżności (Lokalnie vs Online): {len(mismatch_local_vs_online)}")

if not mismatch_local_vs_online.empty:
    print("\nPrzykłady rozbieżności (Twoja lista vs MyGene):")
    print(mismatch_local_vs_online[['Name', 'Description', 'is_protein_mygene', 'is_protein_ensembl_local']].head(10))

Pobieranie danych dla 74628 genów z MyGene.info...


35 input query terms found dup hits:	[('ENSG00000291072', 2), ('ENSG00000228044', 2), ('ENSG00000226506', 2), ('ENSG00000261600', 2), ('E
53 input query terms found no hit:	['ENSG00000304412', 'ENSG00000309633', 'ENSG00000288982', 'ENSG00000300721', 'ENSG00000300484', 'ENS



--- Statystyki Porównawcze ---
Zgodność z MyGene (Protein): 19359
Zgodność z Twoją listą: 19515
Liczba rozbieżności (Lokalnie vs Online): 904

Przykłady rozbieżności (Twoja lista vs MyGene):
                Name Description  is_protein_mygene  is_protein_ensembl_local
116  ENSG00000224870  MRPL20-AS1              False                      True
132  ENSG00000215014   SSU72-AS1              False                      True
209  ENSG00000177133   PRDM16-DT               True                     False
418  ENSG00000179840  PIK3CD-AS1              False                      True
565  ENSG00000179412    HNRNPCL4               True                     False
566  ENSG00000204505     PRAMEF9               True                     False
571  ENSG00000270601     PRAMEF5               True                     False
575  ENSG00000237700    PRAMEF33               True                     False
609  ENSG00000204464  TMEM51-AS2              False                      True
766  ENSG00000211454       A

In [17]:
# 4. Połączenie z df_gtex
# MyGene zwraca DataFrame, gdzie index to oryginalne zapytanie (ID)
mismatch = mismatch.merge(records[['is_protein_mygene']], 
                        left_on='Name', 
                        right_index=True, 
                        how='left')

# 5. Dodanie Twoich poprzednich kolumn (na podstawie lokalnych list)
mismatch['is_protein_ensembl_local'] = mismatch['Name'].isin(ensembl_list)

# 6. Porównanie: Lokalna lista vs MyGene Online
mismatch_local_vs_online = mismatch[mismatch['is_protein_mygene'] != mismatch['is_protein_ensembl_local']]

print(f"\n--- Statystyki Porównawcze ---")
print(f"Zgodność z MyGene (Protein): {mismatch['is_protein_mygene'].sum()}")
print(f"Zgodność z Twoją listą: {mismatch['is_protein_ensembl_local'].sum()}")
print(f"Liczba rozbieżności (Lokalnie vs Online): {len(mismatch_local_vs_online)}")

if not mismatch_local_vs_online.empty:
    print("\nPrzykłady rozbieżności (Twoja lista vs MyGene):")
    print(mismatch_local_vs_online[['Name', 'Description', 'is_protein_mygene', 'is_protein_ensembl_local']].head(10))


--- Statystyki Porównawcze ---
Zgodność z MyGene (Protein): 1321
Zgodność z Twoją listą: 1716
Liczba rozbieżności (Lokalnie vs Online): 495

Przykłady rozbieżności (Twoja lista vs MyGene):
                 Name      Description  is_protein_mygene  \
116   ENSG00000224870       MRPL20-AS1              False   
132   ENSG00000215014        SSU72-AS1              False   
418   ENSG00000179840       PIK3CD-AS1              False   
566   ENSG00000204505          PRAMEF9               True   
571   ENSG00000270601          PRAMEF5               True   
609   ENSG00000204464       TMEM51-AS2              False   
916   ENSG00000249087       ZNF436-AS1              False   
2109  ENSG00000284686  ENSG00000284686              False   
2298  ENSG00000248458  ENSG00000248458              False   
2378  ENSG00000197568      ANKRD13C-DT              False   

      is_protein_ensembl_local  
116                       True  
132                       True  
418                       True  
566   